In [ ]:
import torch
import pandas as pd
import numpy as np
import re
import time
import multiprocessing as mp
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Hardware info ──
NUM_CORES = mp.cpu_count()  # 12 logical cores
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  : {torch.__version__}  (CUDA build: {torch.version.cuda})")
print(f"CPU cores: {NUM_CORES}")
print(f"Device   : {DEVICE} {'(' + torch.cuda.get_device_name(0) + ')' if DEVICE.type == 'cuda' else '(no GPU)'}")
if DEVICE.type == 'cuda':
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ CUDA not available — install with:")
    print("  conda install pytorch torchvision torchaudio pytorch-cuda=12.4 -c pytorch -c nvidia")

In [ ]:
df = pd.read_csv('cleaned_news.csv')
display(df.head())

In [ ]:
nifty_df = pd.read_csv('nifty50_historical_prices.csv')
display(nifty_df.head())

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
nifty_df['Date'] = pd.to_datetime(nifty_df['Date'], errors='coerce')

print(f"Datatype of df['date']: {df['date'].dtype}")
print(f"Datatype of nifty_df['Date']: {nifty_df['Date'].dtype}")

In [ ]:
df['date'] = df['date'].dt.date

print("First row of df after stripping time:")
display(df.head(1))
print(f"Datatype of df['date'] after stripping time: {df['date'].dtype}")

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# ── Worker function for multiprocessing (must be at module-level scope) ──
def _clean_one(text):
    """Clean a single text string: lowercase → remove punct → lemmatize → drop stopwords."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = ' '.join(lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words)
    return text

# ── Parallel text cleaning using all CPU cores ──
t0 = time.perf_counter()
texts = df['news'].tolist()

with mp.Pool(NUM_CORES) as pool:
    cleaned = list(tqdm(pool.imap(_clean_one, texts, chunksize=512),
                        total=len(texts), desc="Cleaning text"))

df['cleaned_news'] = cleaned
elapsed = time.perf_counter() - t0
print(f"\n✓ Cleaned {len(df):,} articles in {elapsed:.1f}s using {NUM_CORES} cores")

In [ ]:
print("Original News Sample:")
display(df[['news', 'cleaned_news']].head(5))

In [ ]:
df['cleaned_news_len'] = df['cleaned_news'].apply(len)

average_title_length = df['cleaned_news_len'].mean()
longest_title_length = df['cleaned_news_len'].max()

print(f"Average cleaned news text length: {average_title_length:.2f} characters")
print(f"Longest cleaned news text length: {longest_title_length} characters")

In [ ]:
df = df.drop(columns=['author', 'url', 'title'])
print("DataFrame after dropping 'author' and 'url' columns:")
display(df.head())

In [ ]:
unique_tickers = nifty_df['Ticker'].unique()
num_unique_tickers = len(unique_tickers)

print(f"Unique tickers in nifty_df: {unique_tickers}")
print(f"Number of unique tickers: {num_unique_tickers}")

In [ ]:
nifty_df['Ticker'] = nifty_df['Ticker'].str.lower()

print("DataFrame with Ticker column in lowercase:")
display(nifty_df.head())

In [ ]:
print("Value counts for 'date' column:")
display(df['date'].value_counts().head(10))

In [ ]:
df_concatenated = df.groupby('date')['cleaned_news'].apply(lambda x: ' '.join(x)).reset_index()

print("DataFrame with concatenated news per date:")
display(df_concatenated.head())
print(f"New shape of DataFrame: {df_concatenated.shape}")

In [ ]:
df_concatenated['cleaned_news_len'] = df_concatenated['cleaned_news'].apply(len)

average_concatenated_news_length = df_concatenated['cleaned_news_len'].mean()
longest_concatenated_news_length = df_concatenated['cleaned_news_len'].max()

print(f"Average cleaned concatenated news text length: {average_concatenated_news_length:.2f} characters")
print(f"Longest cleaned concatenated news text length: {longest_concatenated_news_length} characters")

In [ ]:
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Load FinBERT with optimizations ──
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
model.eval()  # Turn off dropout etc.

# Move to GPU with FP16 (half precision) → 2x faster inference, half VRAM
if DEVICE.type == 'cuda':
    model = model.half().to(DEVICE)   # FP16
    print("✓ Model loaded on GPU with FP16")
else:
    model = model.to(DEVICE)
    print("✓ Model loaded on CPU (FP32)")

# torch.compile() for PyTorch 2.x — fuses ops for ~20-30% speedup
try:
    model = torch.compile(model, mode="reduce-overhead")
    print("✓ torch.compile() applied (reduce-overhead mode)")
except Exception as e:
    print(f"⚠ torch.compile() skipped: {e}")

print(f"Label mapping: {model.config.id2label}")

In [ ]:
# ── Configuration ──
# RTX 3050 4GB: FP16 FinBERT ≈ 130MB → plenty of room for large batches
# On CPU: keep batch smaller to avoid memory pressure
WINDOW_SIZE = 512
STRIDE = 256
GPU_BATCH = 64 if DEVICE.type == 'cuda' else 16

print(f"Window: {WINDOW_SIZE} tokens, Stride: {STRIDE}, Batch: {GPU_BATCH}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  OPTIMIZED FinBERT SENTIMENT — Flat-batch with sliding window
# ══════════════════════════════════════════════════════════════════════
#
# Strategy:
#   1. Tokenize every row's text with sliding window → variable-length chunks
#   2. Flatten ALL chunks into one big tensor (track row→chunk mapping)
#   3. Single batched forward pass over the whole tensor on GPU (FP16)
#   4. Aggregate per-row scores by averaging chunk probabilities
#
# This avoids Python-level per-row loops during inference.
# ──────────────────────────────────────────────────────────────────────

texts = df_concatenated['cleaned_news'].tolist()
n_rows = len(texts)

# ── Step 1: Tokenize all texts (CPU) with sliding window ──
t0 = time.perf_counter()

all_input_ids = []
all_attn_masks = []
row_indices = []  # maps each chunk back to its source row

for idx, text in enumerate(tqdm(texts, desc="Tokenizing")):
    if not text or len(text.strip()) == 0:
        continue
    enc = tokenizer(
        text,
        max_length=WINDOW_SIZE,
        truncation=True,
        stride=STRIDE,
        return_overflowing_tokens=True,
        padding="max_length",
        return_tensors="pt",
    )
    n_chunks = enc['input_ids'].size(0)
    all_input_ids.append(enc['input_ids'])
    all_attn_masks.append(enc['attention_mask'])
    row_indices.extend([idx] * n_chunks)

# Concatenate into single tensors
all_input_ids  = torch.cat(all_input_ids, dim=0)
all_attn_masks = torch.cat(all_attn_masks, dim=0)
row_indices    = np.array(row_indices)

total_chunks = all_input_ids.size(0)
tok_time = time.perf_counter() - t0
print(f"\n✓ Tokenized {n_rows} rows → {total_chunks:,} chunks in {tok_time:.1f}s")

# ── Step 2: Batched GPU inference ──
t1 = time.perf_counter()

# Pre-allocate output array [total_chunks, 3]
chunk_probs = np.empty((total_chunks, 3), dtype=np.float32)

with torch.inference_mode():  # faster than torch.no_grad()
    for start in tqdm(range(0, total_chunks, GPU_BATCH), desc="Inference"):
        end = min(start + GPU_BATCH, total_chunks)
        b_ids  = all_input_ids[start:end].to(DEVICE)
        b_mask = all_attn_masks[start:end].to(DEVICE)

        logits = model(b_ids, attention_mask=b_mask).logits
        probs  = torch.nn.functional.softmax(logits, dim=-1)
        chunk_probs[start:end] = probs.float().cpu().numpy()

inf_time = time.perf_counter() - t1
print(f"\n✓ Inference on {total_chunks:,} chunks in {inf_time:.1f}s "
      f"({total_chunks / inf_time:.0f} chunks/s)")

# ── Step 3: Aggregate per-row (average chunk scores) ──
# FinBERT labels: 0=positive, 1=negative, 2=neutral
neg_scores = np.zeros(n_rows, dtype=np.float32)
neu_scores = np.full(n_rows, 1.0, dtype=np.float32)  # default neutral
pos_scores = np.zeros(n_rows, dtype=np.float32)

for row_idx in range(n_rows):
    mask = row_indices == row_idx
    if mask.any():
        row_probs = chunk_probs[mask]  # shape [n_chunks_for_row, 3]
        pos_scores[row_idx] = row_probs[:, 0].mean()
        neg_scores[row_idx] = row_probs[:, 1].mean()
        neu_scores[row_idx] = row_probs[:, 2].mean()

df_concatenated['neg_score'] = neg_scores
df_concatenated['neu_score'] = neu_scores
df_concatenated['pos_score'] = pos_scores

total_time = time.perf_counter() - t0
print(f"\n{'='*50}")
print(f"TOTAL: {total_time:.1f}s  (tokenize: {tok_time:.1f}s + inference: {inf_time:.1f}s)")
print(f"{'='*50}")
display(df_concatenated[['date', 'neg_score', 'neu_score', 'pos_score']].head(10))

In [ ]:
df_concatenated['date'] = pd.to_datetime(df_concatenated['date'])
merged_df = pd.merge(nifty_df, df_concatenated, left_on='Date', right_on='date', how='left')

print("Merged DataFrame (nifty_df left-joined with df_concatenated) head:")
display(merged_df.head())
print(f"New shape of merged_df: {merged_df.shape}")

In [ ]:
#merged_df = merged_df.drop(columns=['date'])
print("DataFrame after discarding 'date' column:")
display(merged_df.head(115))

In [ ]:
out_path = 'merged_financial_data_news.csv'
merged_df.to_csv(out_path, index=False)
print(f"✓ Saved merged DataFrame → {out_path}  ({len(merged_df):,} rows)")